# PRJEB30386 — эталон после аннотации → PCR1 → фрагментация → PCR2 → NovaSeq PE150

Биологический эталон строится только из `post_annotation_filtered/airr_pass`. Симуляция сохраняет наблюдаемую нуклеотидную последовательность V…J, включая реальные SHM, использует численность UMI как число исходных копий, добавляет общую модель ошибок InSilicoSeq NovaSeq.

**Фрагментация:** `uniform single random-cut`.

**Сценарий фиксированных кратностей ПЦР:** ПЦР1 увеличивает общее число молекул ровно в 20 раз, ПЦР2 — в 3,5 раза; новые копии раздаются случайно пропорционально числу копий. Каждая молекула порции фрагментируется отдельно. Бюджета ридов нет: каждая копия, отобранная на секвенирование, даёт одну пару.


## Выходные данные

Имя ветки собирается в ячейке параметров из базового имени, режима эталона, `READ_PAIRS_PER_SELECTED_COPY`, кратностей ПЦР, отбора по длине и параметров ПЦР, например:

`results/PRJEB30386/simulated/<base>_amp_umicons_rpc1_ss200-350h25_pcr1x20_pcr2x3.5_seq1e-3_al1e-1/`

Каталог `qc/` содержит контроль качества симуляции.

## 1. Окружение

In [ ]:
import os, sys, sysconfig, subprocess, time, gzip, csv, math, re, shutil, json, hashlib
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np

_ENV_CANDIDATES = [
    os.environ.get("BCR_ENV", ""),
    os.environ.get("CONDA_PREFIX", ""),
    "/data/user/epishkin/conda/envs/bcr_env",
    "/opt/conda/envs/bcr_env",
    "/Users/epishkin/mamba/envs/bcr_env",
]
_CONDA_ENV = next((p for p in _ENV_CANDIDATES if p and os.path.isdir(str(Path(p) / "bin"))), None)
if not _CONDA_ENV:
    raise FileNotFoundError("bcr_env not found; activate it or set BCR_ENV")

os.environ["PATH"] = str(Path(_CONDA_ENV) / "bin") + ":" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
for _site in [
    str(Path(_CONDA_ENV) / "lib/python3.11/site-packages"),
    str(Path(_CONDA_ENV) / "lib/python3.12/site-packages"),
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)

for tool in ("iss",):
    p = shutil.which(tool)
    if not p:
        raise RuntimeError(f"Required tool not found: {tool}")
    print(f"{tool}: {p}")
print("numpy:", np.__version__)


## 2. Параметры

In [ ]:
BRANCH_NAME = "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut"

# Эталон, входная порция библиотеки и глубина
# amplicon_with_primers: физический ампликон PCR1 = V-leader праймер + склеенный
#   рид (остаток leader, V…J, C-фрагмент) + revcomp(C-праймер). Праймеры вырезаны
#   на стадии pr_trimmed, их имена восстанавливаются из заголовков pr_trimmed.
# vj_only: эталон обрезан ровно до V…J.
TRUTH_TEMPLATE_MODE = "amplicon_with_primers"
# umi_consensus: каждый UMI = одна исходная молекула; её V…J — поколоночный консенсус
#   ридов UMI модального локуса и модальной длины (>= MIN_UMI_CONSENSUS_READS ридов,
#   без ничьих). UMI без консенсуса (1 рид, ничья, мало ридов одной длины) симулируются
#   самым частым ридом, но помечаются verified=False и не входят в оценку сборщиков.
UMI_COLLAPSE_MODE = "umi_consensus"
MIN_UMI_CONSENSUS_READS = 2

# Бюджета пар нет: каждая копия, отобранная из пула ПЦР2 на секвенирование, даёт одну пару ридов.
READ_PAIRS_PER_SELECTED_COPY = 1

# Отбор по длине после фрагментации: задаётся явным диапазоном ниже.
# Внутри диапазона вероятность отбора 1; вне него она уменьшается вдвое на каждые h нт.
SIZE_SELECTION_RANGE = (200, 350)
SIZE_SELECTION_HALF_DISTANCE_NT = 25.0
# ПЦР без циклов и эффективности: общее число молекул растёт ровно в заданное число раз,
# новые копии раздаются случайно пропорционально числу копий шаблона (ПЦР1) или вида куска (ПЦР2).
PCR1_FOLD = 20
PCR2_FOLD = 3.5
PCR2_TO_SEQUENCING_FRACTION = 1e-3  # фиксированная доля копий PCR2 на секвенирование

# Доля копий PCR1, случайно отбираемая в точном общем количестве.
PCR1_TO_FRAGMENTATION_FRACTION = 0.1  # фиксированная доля PCR1 для фрагментации

_TRUTH_TAG = {"amplicon_with_primers": "amp", "vj_only": "vj"}[TRUTH_TEMPLATE_MODE]
assert UMI_COLLAPSE_MODE == "umi_consensus"
_TRUTH_TAG += "_umicons"
BRANCH_NAME = f"{BRANCH_NAME}_{_TRUTH_TAG}_rpc{READ_PAIRS_PER_SELECTED_COPY}_ss{SIZE_SELECTION_RANGE[0]}-{SIZE_SELECTION_RANGE[1]}h{SIZE_SELECTION_HALF_DISTANCE_NT:g}"
# Кратности ПЦР и доли библиотеки входят в имя ветки.
BRANCH_NAME += f"_pcr1x{PCR1_FOLD:g}_pcr2x{PCR2_FOLD:g}_seq1e-3_al1e-1"
SOURCE_STAGE = "post_annotation_filtered"

VOLUME = Path(os.environ.get("BCR_VOLUME", "/data/user/epishkin"))
if not (VOLUME / "results/PRJEB30386").exists():
    found = None
    for start in (Path.cwd().resolve(), Path("/Users/epishkin/workspace/bcr-assembler")):
        for candidate in (start, *start.parents):
            if (candidate / "results/PRJEB30386").exists() and (candidate / "scripts").is_dir():
                found = candidate
                break
        if found:
            break
    if not found:
        raise FileNotFoundError("Cannot locate bcr-assembler root; set BCR_VOLUME")
    VOLUME = found

DATASET = "PRJEB30386"
SOURCE_RUNS = ["ERR3004229", "ERR3004230", "ERR3004231", "ERR3004232"]
RUN_LOCUS = {
    "ERR3004229": "IGH",
    "ERR3004230": "IGH",
    "ERR3004231": "IGK",
    "ERR3004232": "IGL",
}
RUN_LIBRARY = {
    "ERR3004229": "IgM",
    "ERR3004230": "IgG",
    "ERR3004231": "IgK",
    "ERR3004232": "IgL",
}
MIXED_SAMPLE = "PRJEB30386_all_chains"
SAMPLES = [MIXED_SAMPLE]

DATASET_DIR = VOLUME / "results" / DATASET
POST_FILTER_DIR = DATASET_DIR / SOURCE_STAGE
TRUTH_AIRR_DIR = POST_FILTER_DIR / "airr_pass"
FILTERED_FASTQ_DIR = POST_FILTER_DIR / "fastq"
FILTER_SUMMARY_PATH = POST_FILTER_DIR / "filter_summary.json"

RAW_FASTQ_DIR = VOLUME / "raw" / DATASET
PR_FASTQ_DIR = DATASET_DIR / "pr_trimmed" / "fastq"

if not FILTER_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"Post-annotation filter summary not found: {FILTER_SUMMARY_PATH}. "
        "Run filter_human_post_annotation.ipynb first."
    )
FILTER_SUMMARY = json.loads(FILTER_SUMMARY_PATH.read_text())
FILTER_RULES = FILTER_SUMMARY["filters"]

assert FILTER_SUMMARY.get("dataset") == DATASET
assert FILTER_RULES["require_expected_locus"] == RUN_LOCUS
MIN_VJ_SPAN_NT = int(FILTER_RULES["min_vj_span_nt"])
MIN_V_IDENTITY_PERCENT = float(FILTER_RULES["min_v_identity_percent"])
MIN_J_IDENTITY_PERCENT = float(FILTER_RULES["min_j_identity_percent"])
MAX_V_SUPPORT_EVALUE = float(FILTER_RULES["max_v_support_evalue"])
MAX_J_SUPPORT_EVALUE = float(FILTER_RULES["max_j_support_evalue"])

TARGET_READ_LENGTH_NT = 150

OUT_BASE = DATASET_DIR / "simulated" / BRANCH_NAME
TRUTH_DIR = OUT_BASE / "00_primary_truth"
PCR1_DIR = OUT_BASE / "01_pcr1"
FRAGMENTATION_DIR = OUT_BASE / "02_fragmentation"
PCR2_DIR = OUT_BASE / "03_pcr2"
ALLOCATION_DIR = OUT_BASE / "04_read_allocation"
FASTQ_NATIVE_DIR = OUT_BASE / "05_fastq_native"
FASTQ_DIR = OUT_BASE / "06_fastq_pe150"
MODEL_DIR = OUT_BASE / "model"
LOGS_DIR = OUT_BASE / "logs"
QC_DIR = OUT_BASE / "qc"
TEMPLATES_DIR = TRUTH_DIR

for d in (
    TRUTH_DIR, PCR1_DIR, FRAGMENTATION_DIR, PCR2_DIR, ALLOCATION_DIR,
    FASTQ_NATIVE_DIR, FASTQ_DIR, MODEL_DIR, LOGS_DIR, QC_DIR,
):
    d.mkdir(parents=True, exist_ok=True)

# Число CPU для генератора InSilicoSeq.
ISS_GENERATOR_CPUS = 8
# базовое зерно; зерно образца = SEED + сумма кодов символов его имени
SEED = 42
# FORCE и CLEAN_OLD_OUTPUTS вместе разрешают удалить содержимое ветки и пересчитать её
FORCE = False
CLEAN_OLD_OUTPUTS = False

# Дополнительные требования симуляции к эталонным данным после
# основной фильтрации аннотаций.
# записи с неоднозначными N исключаются
DROP_SEQUENCES_WITH_N = True
# записи без корректной 20-нт UMI исключаются
REQUIRE_VALID_20NT_UMI = True
# исходные копии шаблона = число его UMI
STARTING_COPIES_MODE = "unique_umis"


# Фрагментация и отбор библиотеки.
#
# Из пула PCR1 случайно выбирается заданная доля копий; каждая выбранная
# молекула разрезается на две части, суммарная длина которых равна длине шаблона.
# amplicon: InSilicoSeq читает готовые фрагменты и сам не фрагментирует
SEQUENCE_TYPE = "amplicon"
FRAGMENTATION_MODEL = "uniform_single_cut"


# Отбор по длине после разрезания: диапазон SIZE_SELECTION_RANGE либо окно
# InSilicoSeq требует фрагмент длиннее рида модели NovaSeq (151 нт).
MIN_FRAGMENT_LENGTH_FOR_SEQUENCING_NT = 152

FRAGMENTATION_MODEL_PROVENANCE = {
    "model": FRAGMENTATION_MODEL,
    "pcr1_aliquot": "fixed-size random sample without replacement; exact total is round(fraction × PCR1 pool)",
    "pcr1_aliquot_fraction": PCR1_TO_FRAGMENTATION_FRACTION,
    "fragment_pool": "every aliquot molecule is cut individually; each daughter kept with p(length); identical (template, start, end) pieces pooled into one species",
    "pcr_model": "fixed total fold; new copies assigned at random proportional to current copies",
    "pcr1_fold": PCR1_FOLD,
    "pcr2_fold": PCR2_FOLD,
    "pcr2_sequencing_sample": "fixed-size random sample without replacement; exact total is round(fraction × PCR2 pool)",
    "pcr2_sequencing_fraction": PCR2_TO_SEQUENCING_FRACTION,
    "breakpoints": "exactly one uniformly sampled inter-base bond per library-input molecule",
    "daughter_fragments": "both physical daughters are created before size selection",
    "size_selection": {
        "range_nt": SIZE_SELECTION_RANGE,
        "half_distance_nt": SIZE_SELECTION_HALF_DISTANCE_NT,
        "min_sequencable_nt": MIN_FRAGMENT_LENGTH_FOR_SEQUENCING_NT,
    },
}



SEQUENCING_ERROR_MODEL_PROVENANCE = {
    "type": "InSilicoSeq bundled NovaSeq profile",
    "conditioned_on_post_annotation_filter": False,
}

print("SOURCE STAGE:", SOURCE_STAGE)
print("TRUTH AIRR:", TRUTH_AIRR_DIR)
print("BRANCH:", BRANCH_NAME)
print("OUT_BASE:", OUT_BASE)
print("filter rules:", FILTER_RULES)


In [ ]:
# Очистка сгенерированных артефактов в целевой ветви
# Каталог выходных данных:
#   results/PRJEB30386/simulated/<имя_ветки>/
#
# При FORCE=True и CLEAN_OLD_OUTPUTS=True содержимое целевых
# выходных каталогов удаляется и создаётся заново. Исходные данные
# за пределами OUT_BASE не изменяются.

def clean_previous_generated_outputs():
    if not (FORCE and CLEAN_OLD_OUTPUTS):
        print("Cleanup disabled.")
        return

    generated_dirs = [TRUTH_DIR,PCR1_DIR,FRAGMENTATION_DIR,PCR2_DIR,ALLOCATION_DIR,FASTQ_NATIVE_DIR,FASTQ_DIR,MODEL_DIR,QC_DIR,LOGS_DIR]

    for directory in generated_dirs:
        directory = Path(directory)
        if not directory.exists():
            continue
        for path in directory.iterdir():
            if path.is_file() or path.is_symlink():
                path.unlink()
            elif path.is_dir():
                shutil.rmtree(path)

    for directory in generated_dirs:
        Path(directory).mkdir(parents=True, exist_ok=True)

    print(f"Cleaned previous generated outputs under: {OUT_BASE}")

clean_previous_generated_outputs()


## 3. Вспомогательные функции

In [ ]:
def open_text(path, mode="rt"):
    return gzip.open(path, mode) if str(path).endswith(".gz") else open(path, mode)

def iter_fastq(path):
    with open_text(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                return
            seq = h.readline().rstrip("\n\r")
            plus = h.readline()
            qual = h.readline()
            if not plus or not qual:
                raise ValueError(f"Truncated FASTQ: {path}")
            yield seq

def count_fastq(path):
    return sum(1 for _ in iter_fastq(path))

def iter_fasta(path):
    with open(path) as h:
        name, chunks = None, []
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:], []
            else:
                chunks.append(line)
        if name is not None:
            yield name, "".join(chunks)

def run_with_heartbeat(cmd, log_path, heartbeat=30, shell=False):
    t0 = time.time()
    with open(log_path, "w") as log_h:
        proc = subprocess.Popen(cmd, stdout=log_h, stderr=subprocess.STDOUT, text=True, shell=shell)
        label = cmd if shell else " ".join(map(str, cmd))
        print(f"[run] {label}\n  pid={proc.pid} log={log_path}")
        while proc.poll() is None:
            print(f"  running: elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(heartbeat)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}); see {log_path}")
    print(f"done in {(time.time()-t0)/60:.1f} min")

def sample_seed(sample, extra=0):
    return int(SEED + extra + sum((i + 1) * ord(ch) for i, ch in enumerate(sample)))

## 4. Построение эталона: ампликон PCR1 с учётом UMI

`post_annotation_filtered/airr_pass` служит эталонным входом. Условия фильтра повторно проверяются утверждениями. Дополнительно исключаются непригодные UMI и записи с N.

- **Ключ схлопывания** — (локус, наблюдаемая последовательность V…J); реальные SHM сохраняются.
- **`UMI_COLLAPSE_MODE = "umi_consensus"`**: каждый UMI считается одной исходной молекулой. Консенсус V…J строится, если есть не меньше `MIN_UMI_CONSENSUS_READS` ридов однозначно преобладающего локуса и длины. В каждой позиции выбирается самая частая база. **Ничья** — два или более варианта имеют одинаковую максимальную поддержку: это может произойти при выборе локуса, длины или базы в позиции. Такая UMI-молекула не считается проверенной (`verified=False`); вместо консенсуса берётся самый частый рид. Если он отличается одной заменой от проверенного консенсуса того же локуса, он переносится на этот консенсус как вероятная ошибка исходного секвенирования. Шаблон без проверенной UMI остаётся в библиотеке, но не входит в оценку сборщиков. CDR3 (`junction`) записывается в `template_qc.tsv`.
- **`TRUTH_TEMPLATE_MODE = "amplicon_with_primers"`**: шаблон = V-leader праймер + склеенный рид (остаток leader, V…J, C-фрагмент) + revcomp(C-праймер). Праймеры были вырезаны `MaskPrimers --mode cut`; имена праймеров берутся из заголовков `pr_trimmed` (AssemblePairs их не перенёс), последовательности — из `technical_sequences.json`. UMI и Illumina-хвосты не включаются. Координаты V…J записаны в `vj_start`/`vj_end` (0-based, half-open) — именно этот интервал является эталоном для оценки сборщиков.


In [ ]:
# Коды неоднозначных нуклеотидов IUPAC; ниже каждый такой символ заменяется одной допустимой буквой.
_IUPAC = {"A": "A", "C": "C", "G": "G", "T": "T", "R": "AG", "Y": "CT", "S": "CG", "W": "AT",
          "K": "GT", "M": "AC", "B": "CGT", "D": "AGT", "H": "ACT", "V": "ACG", "N": "ACGT"}
# Таблица комплементарности оснований для построения обратного комплемента.
_RC = str.maketrans("ACGT", "TGCA")

def _revcomp(s):
    return s.translate(_RC)[::-1]

def _truthy(x):
    return str(x).strip().lower() in {"t", "true", "1", "yes"}

def _falsey(x):
    return str(x).strip().lower() in {"f", "false", "0", "no"}

def _as_int(x):
    try:
        return int(x)
    except (TypeError, ValueError):
        return None

def _as_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def _barcode(sequence_id):
    m = re.search(r"(?:^|\|)BARCODE=([^|\s]+)", sequence_id or "")
    return m.group(1) if m else None

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as src:
        for chunk in iter(lambda: src.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _find_technical_sequences():
    rel = Path("docs/datasets/PRJEB30386/technical_sequences.json")
    cwd = Path.cwd().resolve()
    for root in (VOLUME, cwd, *cwd.parents, Path("/Users/epishkin/workspace/bcr-assembler")):
        if (root / rel).exists():
            return root / rel
    raise FileNotFoundError(f"{rel} not found; needed for primer sequences")

def load_gene_specific_primers():
    """Gene-specific части праймеров библиотеки (без Illumina-хвоста и C-side UMI).

    Совпадают со словарями V_LEADER_PRIMERS / CONSTANT_PRIMERS в primer_trim_human.ipynb.
    """
    path = _find_technical_sequences()
    out = {}
    for p in json.loads(path.read_text())["supplemental_table_2_library_primers"]:
        tail = p["sequence"].split("TCTTCCGATCT", 1)
        if len(tail) != 2 or not tail[1]:
            continue  # Read2U и прочие чисто технические олиго
        out[p["name"]] = tail[1][20:] if "_Ig" in p["name"] else tail[1]
    return out, path

def _pr_trimmed_primer_names(run):
    """read id -> [V primer, C primer] из заголовков pr_trimmed (AssemblePairs их не перенёс)."""
    names = {}
    for mate, slot in ((1, 0), (2, 1)):
        with gzip.open(PR_FASTQ_DIR / f"{run}_{mate}.pr.fastq.gz", "rt") as h:
            for i, line in enumerate(h):
                if i % 4:
                    continue
                rid = line[1:].split(None, 1)[0].split("|", 1)[0]
                m = re.search(r"\|PRIMER=([^|\s]+)", line)
                names.setdefault(rid, [None, None])[slot] = m.group(1) if m else None
    return names

if TRUTH_TEMPLATE_MODE == "amplicon_with_primers":
    PRIMERS, TECHNICAL_SEQUENCES_PATH = load_gene_specific_primers()
    print("gene-specific primers:", len(PRIMERS), "from", TECHNICAL_SEQUENCES_PATH)
else:
    PRIMERS, TECHNICAL_SEQUENCES_PATH = {}, None

FILTER_SUMMARY_SHA256 = _sha256(FILTER_SUMMARY_PATH)
PROVENANCE_PATH = TRUTH_DIR / "source_provenance.json"

def _validate_post_filter_row(r, run):
    """Останавливает работу, если airr_pass не соответствует записанным условиям фильтра."""
    problems = []
    expected_locus = RUN_LOCUS[run]

    v_start = _as_int(r.get("v_sequence_start"))
    j_end = _as_int(r.get("j_sequence_end"))
    vgs = _as_int(r.get("v_germline_start"))
    vi = _as_float(r.get("v_identity"))
    ji = _as_float(r.get("j_identity"))
    vs = _as_float(r.get("v_support"))
    js = _as_float(r.get("j_support"))

    if v_start is None or j_end is None or j_end - v_start + 1 < MIN_VJ_SPAN_NT:
        problems.append("V-J span")
    if vgs != 1:
        problems.append("V 5-prime completeness")
    if not _truthy(r.get("complete_vdj")):
        problems.append("complete_vdj")
    if str(r.get("locus", "")).strip() != expected_locus:
        problems.append("locus")
    if not _truthy(r.get("productive")):
        problems.append("productive")
    if not _falsey(r.get("stop_codon")):
        problems.append("stop_codon")
    if vi is None or vi < MIN_V_IDENTITY_PERCENT:
        problems.append("v_identity")
    if ji is None or ji < MIN_J_IDENTITY_PERCENT:
        problems.append("j_identity")
    if vs is None or vs > MAX_V_SUPPORT_EVALUE:
        problems.append("v_support")
    if js is None or js > MAX_J_SUPPORT_EVALUE:
        problems.append("j_support")

    if problems:
        raise RuntimeError(
            f"{run} airr_pass violates filter_summary contract for "
            f"{r.get('sequence_id')}: {', '.join(problems)}"
        )

def _current_provenance():
    data = {
        "dataset": DATASET,
        "source_stage": SOURCE_STAGE,
        "truth_airr_dir": str(TRUTH_AIRR_DIR),
        "filtered_fastq_dir": str(FILTERED_FASTQ_DIR),
        "filter_summary": str(FILTER_SUMMARY_PATH),
        "filter_summary_sha256": FILTER_SUMMARY_SHA256,
        "filter_rules": FILTER_RULES,
        "simulation_specific_truth_rules": {
            "require_valid_20nt_umi": REQUIRE_VALID_20NT_UMI,
            "drop_sequences_with_N": DROP_SEQUENCES_WITH_N,
            "truth_template_mode": TRUTH_TEMPLATE_MODE,
            "primer_sequences": str(TECHNICAL_SEQUENCES_PATH) if TECHNICAL_SEQUENCES_PATH else None,
            "primer_names_source": str(PR_FASTQ_DIR) if PRIMERS else None,
            "flanks": "самые частые V-праймер, остаток leader, C-фрагмент и C-праймер среди ридов семьи; неоднозначные коды IUPAC разрешаются генератором с фиксированным зерном; UMI в шаблон не входит",
            "vj_coordinates": "template_qc.tsv vj_start/vj_end, 0-based half-open",
            "collapse_key": ["locus", "VJ_sequence"],
            "umi_collapse_mode": UMI_COLLAPSE_MODE,
            "min_umi_consensus_reads": MIN_UMI_CONSENSUS_READS,
            "consensus_tie_policy": "ничья при выборе локуса, длины или основания в любой позиции помечает молекулу UMI как непроверенную",
            "error_correction": "unverified UMI molecule 1 substitution from a verified consensus (same locus and length) -> copy of that consensus",
            "verified": "template has >= 1 UMI whose consensus equals its V-J",
            "junction": "most frequent AIRR junction span among member reads, cut from template V-J",
            "starting_copies_mode": STARTING_COPIES_MODE,
        },
        "branch": BRANCH_NAME,
        "read_pairs_per_selected_copy": READ_PAIRS_PER_SELECTED_COPY,
        "pcr1_fold": PCR1_FOLD,
        "pcr2_fold": PCR2_FOLD,
        "seed": SEED,
        "fragmentation_model": FRAGMENTATION_MODEL_PROVENANCE,
        "sequencing_error_model": SEQUENCING_ERROR_MODEL_PROVENANCE,
    }
    return data

def _check_cached_truth_provenance():
    if not PROVENANCE_PATH.exists():
        return False
    old = json.loads(PROVENANCE_PATH.read_text())
    if old.get("filter_summary_sha256") != FILTER_SUMMARY_SHA256:
        raise RuntimeError(
            "post_annotation_filtered/filter_summary.json changed since this "
            "simulation truth was built. Use FORCE=True and CLEAN_OLD_OUTPUTS=True "
            "or choose a new branch."
        )
    if old.get("simulation_specific_truth_rules") != _current_provenance()["simulation_specific_truth_rules"]:
        raise RuntimeError("Truth construction rules changed; use FORCE=True or a new branch.")
    return True

def _consensus(seqs):
    """Поколоночный консенсус ридов одной длины; None при ничьей в любой позиции."""
    if len(set(seqs)) == 1:
        return seqs[0]
    out = []
    for col in zip(*seqs):
        top = Counter(col).most_common(2)
        if len(top) > 1 and top[0][1] == top[1][1]:
            return None
        out.append(top[0][0])
    return "".join(out)

# Одинаковые риды дают ту же последовательность; большинство в одной позиции разрешает разногласие.
assert _consensus(["ACGT", "ACGT"]) == "ACGT"
assert _consensus(["ACGT", "ACGA", "ACGT"]) == "ACGT"
# При равном числе G и T в последней позиции консенсуса нет.
assert _consensus(["ACGT", "ACGA"]) is None


def _umi_molecule(reads):
    """Строит последовательность одной UMI-семьи и отмечает надёжность.

    Возвращает локус, V…J, использованные риды и признак verified. Консенсус
    строится по ридам однозначно преобладающего локуса и длины. Если есть ничья,
    ридов меньше порога или нет однозначного консенсуса, берётся самый частый
    рид и молекула помечается как непроверенная.
    """
    locus_counts = Counter(read[0] for read in reads)
    top_loci = locus_counts.most_common(2)
    locus_tie = len(top_loci) > 1 and top_loci[0][1] == top_loci[1][1]
    locus = max(locus_counts.items(), key=lambda item: (item[1], item[0]))[0]
    reads_in_locus = [read for read in reads if read[0] == locus]

    length_counts = Counter(len(read[1]) for read in reads_in_locus)
    top_lengths = length_counts.most_common(2)
    length_tie = len(top_lengths) > 1 and top_lengths[0][1] == top_lengths[1][1]
    modal_length = max(length_counts.items(), key=lambda item: (item[1], item[0]))[0]
    consensus_reads = [read for read in reads_in_locus if len(read[1]) == modal_length]

    if not locus_tie and not length_tie and len(consensus_reads) >= MIN_UMI_CONSENSUS_READS:
        consensus = _consensus([read[1] for read in consensus_reads])
        if consensus is not None:
            return locus, consensus, consensus_reads, True

    sequence_counts = Counter(read[1] for read in reads_in_locus)
    fallback_sequence = max(sequence_counts.items(), key=lambda item: (item[1], item[0]))[0]
    fallback_reads = [read for read in reads_in_locus if read[1] == fallback_sequence]
    return locus, fallback_sequence, fallback_reads, False
def _one_mismatch(a, b):
    d = 0
    for x, y in zip(a, b):
        if x != y:
            d += 1
            if d > 1:
                return False
    return True

_r = lambda s: ("IGH", s, None, None, "v", "j")
# Соотношение 2:1 даёт однозначный консенсус; одна запись или ничья оснований — непроверенная молекула.
assert _umi_molecule([_r("ACGT"), _r("ACGA"), _r("ACGT")])[1:4:2] == ("ACGT", True)
assert _umi_molecule([_r("ACGT")])[3] is False
assert _umi_molecule([_r("ACGT"), _r("ACGA")])[3] is False
# Равная поддержка двух локусов или двух длин тоже не подтверждает консенсус.
assert _umi_molecule([_r("ACGT"), ("IGK", "ACGT", None, None, "v", "j")])[3] is False
assert _umi_molecule([_r("ACG"), _r("ACGT")])[3] is False


def build_mixed_templates(force=FORCE):
    sample = MIXED_SAMPLE
    out_fa = TEMPLATES_DIR / f"{sample}_templates.fasta"
    out_tsv = TEMPLATES_DIR / f"{sample}_template_qc.tsv"
    audit_path = QC_DIR / "source_truth_audit_by_run.tsv"
    umi_audit_path = QC_DIR / "umi_consensus_audit.tsv"

    if out_fa.exists() and out_tsv.exists() and audit_path.exists() and not force:
        _check_cached_truth_provenance()
        print("[skip] post-filter UMI-consensus templates exist")
        return

    assert REQUIRE_VALID_20NT_UMI, "umi_consensus требует UMI у каждой записи"
    umi_reads = defaultdict(list)  # ключ «запуск:UMI» указывает риды одной исходной молекулы.
    audit = []
    amplicon = TRUTH_TEMPLATE_MODE == "amplicon_with_primers"

    for run in SOURCE_RUNS:
        airr = TRUTH_AIRR_DIR / f"{run}.airr.tsv"
        filtered_fastq = FILTERED_FASTQ_DIR / f"{run}_filtered.fastq.gz"
        if not airr.exists():
            raise FileNotFoundError(f"Filtered AIRR missing: {airr}")
        if not filtered_fastq.exists():
            raise FileNotFoundError(f"Filtered FASTQ missing: {filtered_fastq}")
        primer_names = _pr_trimmed_primer_names(run) if amplicon else {}

        expected_passed = int(FILTER_SUMMARY["samples"][run]["passed"])
        a = {
            "source_run": run,
            "library": RUN_LIBRARY[run],
            "expected_locus": RUN_LOCUS[run],
            "expected_passed_from_filter_summary": expected_passed,
            "airr_pass_rows": 0,
            "simulation_eligible_rows": 0,
            "missing_barcode": 0,
            "invalid_barcode_length": 0,
            "non_acgt_barcode": 0,
            "missing_v_or_j_call": 0,
            "sequence_with_N": 0,
            "missing_primer_annotation": 0,
        }

        with open(airr, newline="") as h:
            for r in csv.DictReader(h, delimiter="\t"):
                a["airr_pass_rows"] += 1
                _validate_post_filter_row(r, run)

                full = (r.get("sequence") or "").upper()
                start = int(r["v_sequence_start"]) - 1
                end = int(r["j_sequence_end"])
                seq = full[start:end]

                if len(seq) < MIN_VJ_SPAN_NT:
                    raise RuntimeError(
                        f"{run}: cropped V-J sequence shorter than filter threshold "
                        f"for {r.get('sequence_id')}"
                    )

                if not r.get("v_call") or not r.get("j_call"):
                    a["missing_v_or_j_call"] += 1
                    continue

                if DROP_SEQUENCES_WITH_N and "N" in (full if amplicon else seq):
                    a["sequence_with_N"] += 1
                    continue

                bc = _barcode(r.get("sequence_id"))
                if not bc:
                    a["missing_barcode"] += 1
                    continue
                if len(bc) != 20:
                    a["invalid_barcode_length"] += 1
                    continue
                if set(bc.upper()) - set("ACGT"):
                    a["non_acgt_barcode"] += 1
                    continue

                if amplicon:
                    vp, c_primer = primer_names.get(r["sequence_id"].split("|", 1)[0], (None, None))
                    if vp not in PRIMERS or c_primer not in PRIMERS:
                        a["missing_primer_annotation"] += 1
                        continue
                    flank = (vp, full[:start], full[end:], c_primer)
                else:
                    flank = None

                # junction = sequence[cdr3_start-4 : cdr3_end+3] (AIRR, координаты с 1), в координатах V…J.
                jspan = ((int(r["cdr3_start"]) - 4 - start, len(r["junction"]))
                         if r.get("junction") and r.get("cdr3_start") else None)
                a["simulation_eligible_rows"] += 1
                umi_reads[run + ":" + bc].append((r["locus"], seq, flank, jspan, r["v_call"], r["j_call"]))

        if a["airr_pass_rows"] != expected_passed:
            raise RuntimeError(
                f"{run}: airr_pass rows={a['airr_pass_rows']:,}, but "
                f"filter_summary passed={expected_passed:,}"
            )
        audit.append(a)
        del primer_names

    # Одна молекула на UMI: консенсус (verified) или самый частый рид (не проверен).
    molecules = []
    umi_stats = Counter()
    for umi, reads in umi_reads.items():
        locus, seq, members, verified = _umi_molecule(reads)
        umi_stats["consensus_verified" if verified else
                  ("single_read" if len(reads) == 1 else "no_consensus")] += 1
        molecules.append((umi.split(":", 1)[0], locus, seq, members, verified))
    del umi_reads

    # Непроверенная молекула, отличающаяся одной заменой от проверенного консенсуса того же
    # локуса, — тот же клон с ошибкой секвенирования (~15% ридов проверенных UMI несут замены):
    # она становится копией этого консенсуса (при нескольких — самого поддержанного).
    support = Counter((l, s) for _, l, s, _, v in molecules if v)
    index = defaultdict(list)
    for l, s in support:
        h = len(s) // 2
        index[(l, len(s), 0, s[:h])].append(s)
        index[(l, len(s), 1, s[h:])].append(s)

    def _verified_neighbour(locus, s):
        L, h, best = len(s), len(s) // 2, None
        for key, lo, hi in (((locus, L, 0, s[:h]), h, L), ((locus, L, 1, s[h:]), 0, h)):
            for p in index.get(key, ()):
                if p != s and _one_mismatch(s[lo:hi], p[lo:hi]) and (
                        best is None or (support[(locus, p)], p) > (support[(locus, best)], best)):
                    best = p
        return best

    groups = {}
    for run, locus, seq, members, verified in molecules:
        if not verified and (locus, seq) not in support:
            parent = _verified_neighbour(locus, seq)
            if parent:
                seq = parent
                umi_stats["unverified_corrected_to_consensus_1nt"] += 1
        g = groups.setdefault((locus, seq), {
            "observed_reads": 0, "umi_copies": 0, "verified_umis": 0,
            "runs": set(), "libraries": set(), "v_calls": set(), "j_calls": set(),
            "flanks": Counter(), "junctions": Counter(),
        })
        g["umi_copies"] += 1
        g["verified_umis"] += int(verified)
        g["observed_reads"] += len(members)
        g["runs"].add(run)
        g["libraries"].add(RUN_LIBRARY[run])
        for _, _, flank, jspan, v_call, j_call in members:
            g["v_calls"].add(v_call)
            g["j_calls"].add(j_call)
            if flank:
                g["flanks"][flank] += 1
            if jspan:
                g["junctions"][jspan] += 1
    del molecules
    print(f"UMI molecules: {dict(umi_stats)}; unique V-J={len(groups):,}; "
          f"verified V-J={sum(g['verified_umis'] > 0 for g in groups.values()):,}")

    ordered = sorted(
        groups.items(),
        key=lambda x: (
            x[0][0],
            -x[1]["umi_copies"],
            -x[1]["observed_reads"],
            x[0][1],
        ),
    )

    iupac_rng = np.random.default_rng(sample_seed(sample, 500))
    def _resolve(p):
        return "".join(c if c in "ACGT" else str(iupac_rng.choice(list(_IUPAC[c]))) for c in p)

    fields = [
        "template_id", "locus", "length", "vj_start", "vj_end", "v_primer", "c_primer",
        "observed_multiplicity", "unique_umis", "verified_umis", "verified", "junction",
        "source_runs", "libraries", "v_calls", "j_calls",
    ]
    with open(out_fa, "w") as fa, open(out_tsv, "w", newline="") as ts:
        w = csv.DictWriter(ts, fieldnames=fields, delimiter="\t")
        w.writeheader()
        for i, ((locus, seq), g) in enumerate(ordered, 1):
            tid = f"{sample}_{locus}_tpl_{i:07d}"
            if amplicon:
                vp, lead, cstub, c_primer = g["flanks"].most_common(1)[0][0]
                left = _resolve(PRIMERS[vp]) + lead
                template = left + seq + cstub + _revcomp(_resolve(PRIMERS[c_primer]))
            else:
                vp = c_primer = ""
                left, template = "", seq
            jo, jl = g["junctions"].most_common(1)[0][0] if g["junctions"] else (-1, 0)
            fa.write(f">{tid}\n{template}\n")
            w.writerow({
                "template_id": tid,
                "locus": locus,
                "length": len(template),
                "vj_start": len(left),
                "vj_end": len(left) + len(seq),
                "v_primer": vp,
                "c_primer": c_primer,
                "observed_multiplicity": g["observed_reads"],
                "unique_umis": g["umi_copies"],
                "verified_umis": g["verified_umis"],
                "verified": g["verified_umis"] > 0,
                "junction": seq[jo:jo + jl] if 0 <= jo and jo + jl <= len(seq) else "",
                "source_runs": ";".join(sorted(g["runs"])),
                "libraries": ";".join(sorted(g["libraries"])),
                "v_calls": ";".join(sorted(g["v_calls"])),
                "j_calls": ";".join(sorted(g["j_calls"])),
            })

    with open(audit_path, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=list(audit[0]), delimiter="\t")
        w.writeheader()
        w.writerows(audit)
    with open(umi_audit_path, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=["umi_class", "umis"], delimiter="\t")
        w.writeheader()
        w.writerows({"umi_class": k, "umis": v} for k, v in sorted(umi_stats.items()))

    PROVENANCE_PATH.write_text(json.dumps(_current_provenance(), indent=2) + "\n")
    print(
        f"post-filter mixed templates={len(ordered):,}; "
        f"loci={sorted({k[0] for k in groups})}"
    )

build_mixed_templates()


## 5. Сводка контроля качества дедупликации

In [ ]:
q=TEMPLATES_DIR/f"{MIXED_SAMPLE}_template_qc.tsv"; rows=list(csv.DictReader(open(q),delimiter="\t")); loci=sorted({r["locus"] for r in rows}); summary=[]
for locus in loci:
    x=[r for r in rows if r["locus"]==locus]; summary.append({"locus":locus,"unique_templates":len(x),"observed_reads":sum(int(r["observed_multiplicity"]) for r in x),"unique_umis":sum(int(r["unique_umis"]) for r in x),"verified_templates":sum(r["verified"]=="True" for r in x),"verified_umis":sum(int(r["verified_umis"]) for r in x)})
out=QC_DIR/"template_summary_by_locus.tsv"
with open(out,"w",newline="") as h:
    w=csv.DictWriter(h,fieldnames=list(summary[0]),delimiter="\t"); w.writeheader(); w.writerows(summary)
print(*summary,sep="\n"); print("wrote",out)


## 6. Готовая модель ISS NovaSeq

Встроенный KDE-профиль NovaSeq из ISS 2.0.1 создаёт риды длиной 151 nt. После стадий эталон → PCR1 → фрагментация → PCR2 генерируются PE151, затем последовательности и качества детерминированно обрезаются до PE150. Пользовательская модель не обучается.

In [ ]:
import iss
ISS_PROFILE_NAME="NovaSeq"; ISS_NATIVE_READ_LENGTH=151
profile_path=Path(iss.__file__).resolve().parent/"profiles"/ISS_PROFILE_NAME
with np.load(profile_path,allow_pickle=True) as z:
    assert int(z["read_length"])==ISS_NATIVE_READ_LENGTH
    print({"profile":ISS_PROFILE_NAME,"model":str(z["model"].item()),"native_read_length":int(z["read_length"]),"target":TARGET_READ_LENGTH_NT})


## 7. Число ридов

Бюджета пар нет: каждая копия, отобранная из пула ПЦР2 на секвенирование, даёт одну пару ридов PE150. Число пар каждого образца становится известно после раздела 11.


## 8. PCR1 для наблюдаемых шаблонов V…J


In [ ]:
def load_dedup_templates(sample):
    fa=TEMPLATES_DIR/f"{sample}_templates.fasta"; qc=TEMPLATES_DIR/f"{sample}_template_qc.tsv"; meta={r["template_id"]:r for r in csv.DictReader(open(qc),delimiter="\t")}
    return [{"template_id":tid,"sequence":seq,"length":len(seq),"vj_start":int(meta[tid].get("vj_start") or 0),"vj_end":int(meta[tid].get("vj_end") or len(seq)),"observed_multiplicity":int(meta[tid]["observed_multiplicity"]),"unique_umis":int(meta[tid]["unique_umis"]),"locus":meta[tid]["locus"],"verified":meta[tid]["verified"]=="True"} for tid,seq in iter_fasta(fa)]
def starting_copies(row):
    if STARTING_COPIES_MODE=="unique_umis": return max(1,row["unique_umis"])
    if STARTING_COPIES_MODE=="one_per_unique": return 1
    if STARTING_COPIES_MODE=="observed_multiplicity": return row["observed_multiplicity"]
    raise ValueError(STARTING_COPIES_MODE)
def fixed_fold_amplification(copies, fold, rng):
    """Увеличивает общее число копий пула ровно в fold раз (с округлением до целого).

    Каждая новая копия достаётся случайной молекуле пула: вероятность получить её
    пропорциональна текущему числу копий записи (шаблона или вида куска).
    """
    copies = np.asarray(copies, dtype=np.int64)
    total = int(copies.sum())
    new_total = int(np.floor(total * fold + 0.5)) - total
    if total == 0 or new_total <= 0:
        return copies.copy()
    owners = np.searchsorted(
        np.cumsum(copies), rng.integers(0, total, size=new_total), side="right"
    )
    return copies + np.bincount(owners, minlength=len(copies))


def simulate_pcr_pool(sample):
    rng = np.random.default_rng(sample_seed(sample, 1000))
    rows = load_dedup_templates(sample)
    start = np.array([starting_copies(row) for row in rows], dtype=np.int64)
    pcr_copies = fixed_fold_amplification(start, PCR1_FOLD, rng)
    for row, s, copies in zip(rows, start, pcr_copies):
        row.update(starting_copies=int(s), pcr_copies=int(copies))
    return rows


_t = fixed_fold_amplification(np.array([1, 10, 0]), 20, np.random.default_rng(1))
assert int(_t.sum()) == 220 and _t[2] == 0 and _t[0] >= 1 and _t[1] >= 10
_t = fixed_fold_amplification(np.ones(10000, dtype=np.int64), 3.5, np.random.default_rng(1))
assert int(_t.sum()) == 35000 and _t.min() >= 1
print("fixed-fold PCR self-check: OK")


## 9. Фрагментация и отбор по длине

1. Из пула ПЦР1 случайно без возвращения отбирается ровно round(N × PCR1_TO_FRAGMENTATION_FRACTION) молекул.
2. Каждая молекула порции рвётся отдельно по одной из L−1 связей, все связи равновероятны: получаются куски [0, c) и [c, L).
3. Каждый кусок независимо проходит отбор по длине: внутри SIZE_SELECTION_RANGE вероятность 1, вне него 2^(−d/h); куски короче 152 нт не проходят.
4. Одинаковые куски (тот же шаблон, те же start и end) от разных молекул объединяются в один вид куска с числом копий k ≥ 1.


In [ ]:
def _size_selection_probability(length):
    """Возвращает вероятность отбора фрагмента по длине для ПЦР2."""
    length = np.asarray(length, dtype=float)
    low, high = SIZE_SELECTION_RANGE
    distance = np.maximum(np.maximum(low - length, length - high), 0.0)
    probability = 2.0 ** (-distance / SIZE_SELECTION_HALF_DISTANCE_NT)
    return np.where(length < MIN_FRAGMENT_LENGTH_FOR_SEQUENCING_NT, 0.0, probability)


FRAGMENT_ID_TAG = "rc"


def fragment_molecules(molecule_template, molecule_length, rng):
    """Поштучная фрагментация молекул порции и отбор кусков по длине.

    Молекула длины L рвётся по одной из L−1 связей, все связи равновероятны;
    получаются куски [0, c) и [c, L). Каждый кусок независимо проходит отбор
    с вероятностью p(длины). Возвращает шаблон, start и end прошедших кусков.
    """
    molecule_length = np.asarray(molecule_length, dtype=np.int64)
    cut = 1 + (rng.random(len(molecule_length)) * (molecule_length - 1)).astype(np.int64)
    template = np.concatenate([molecule_template, molecule_template])
    start = np.concatenate([np.zeros_like(cut), cut])
    end = np.concatenate([cut, molecule_length])
    keep = rng.random(len(start)) < _size_selection_probability(end - start)
    return template[keep], start[keep], end[keep]


# самопроверка: разрывы внутри молекулы, доля прошедших кусков совпадает с ожидаемой
_L, _n = 450, 200000
_t, _s, _e = fragment_molecules(np.zeros(_n, dtype=np.int64), np.full(_n, _L), np.random.default_rng(1))
_expected = 2 * float(_size_selection_probability(np.arange(1, _L)).mean())
assert _s.min() >= 0 and _e.max() <= _L and ((_e - _s) >= MIN_FRAGMENT_LENGTH_FOR_SEQUENCING_NT).all()
assert abs(len(_s) / _n - _expected) < 0.01
print("fragmentation self-check: OK")


## 10. ПЦР2 и отбор на секвенирование

Общее число кусков после отбора по длине увеличивается ровно в PCR2_FOLD раз: новые копии раздаются случайно пропорционально числу копий каждого вида куска (та же функция `fixed_fold_amplification`, что в ПЦР1). Затем из пула ПЦР2 случайно без возвращения отбирается ровно round(N × PCR2_TO_SEQUENCING_FRACTION) копий; каждая отобранная копия даёт одну пару ридов.


## 11. PCR1 → порция → фрагментация → PCR2 → распределение ридов


In [ ]:
def sample_fixed_size_without_replacement(counts, sample_size, rng):
    """Случайно выбирает заданное число копий из группированного пула.

    Счётчики объединяют копии по шаблонам или видам фрагментов. Сначала
    строится биномиальное приближение, затем случайно корректируется избыток
    или нехватка выбранных копий. В результате получается выборка без
    возвращения с многомерным гипергеометрическим распределением; ограничение
    NumPy для очень больших пулов не используется.
    """
    counts = np.asarray(counts, dtype=np.int64)
    if np.any(counts < 0):
        raise ValueError("Число копий в пуле не может быть отрицательным")
    total = sum(int(value) for value in counts)
    sample_size = int(sample_size)
    if sample_size < 0 or sample_size > total:
        raise ValueError(f"Размер выборки {sample_size} вне диапазона [0, {total}]")
    if sample_size == 0:
        return np.zeros_like(counts)
    if sample_size == total:
        return counts.copy()
    if sample_size > total - sample_size:
        return counts - sample_fixed_size_without_replacement(
            counts, total - sample_size, rng
        )

    probability = sample_size / total
    proposal = np.asarray(rng.binomial(counts, probability), dtype=np.int64)
    proposed_total = sum(int(value) for value in proposal)
    if proposed_total > sample_size:
        correction = sample_fixed_size_without_replacement(
            proposal, proposed_total - sample_size, rng
        )
        return proposal - correction
    if proposed_total < sample_size:
        failures = counts - proposal
        correction = sample_fixed_size_without_replacement(
            failures, sample_size - proposed_total, rng
        )
        return proposal + correction
    return proposal


def sample_fraction_without_replacement(counts, fraction, rng):
    """Случайно выбирает из пула точно заданную долю копий."""
    total = sum(int(value) for value in counts)
    if not 0 <= fraction <= 1:
        raise ValueError("Доля выборки должна быть от 0 до 1")
    target = int(np.floor(total * fraction + 0.5))
    return sample_fixed_size_without_replacement(counts, target, rng)


def write_dict_rows(path, rows, fields):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=fields, delimiter="\t")
        w.writeheader()
        w.writerows({k: r[k] for k in fields} for r in rows)
    tmp.replace(path)

def read_dict_rows(path):
    with open(path, newline="") as h:
        return list(csv.DictReader(h, delimiter="\t"))

def run_pcr1_stage(sample, force=FORCE):
    out = PCR1_DIR / f"{sample}_pcr1_pool.tsv"
    if out.exists() and not force:
        print("[skip] PCR1", out)
        return
    rows = simulate_pcr_pool(sample)
    fields = [
        "template_id", "locus", "length", "observed_multiplicity",
        "unique_umis", "starting_copies", "pcr_copies",
    ]
    write_dict_rows(out, rows, fields)
    print("PCR1 rows:", len(rows), "->", out)


def run_library_stage(sample, force=FORCE):
    """Порция ПЦР1 → поштучная фрагментация и отбор → ПЦР2 → доля на секвенирование → риды."""
    inp = PCR1_DIR / f"{sample}_pcr1_pool.tsv"
    out_counts = ALLOCATION_DIR / f"{sample}_read_counts.tsv"
    out_fa = ALLOCATION_DIR / f"{sample}_selected_fragments.fasta"
    out_qc = ALLOCATION_DIR / f"{sample}_allocation.tsv"
    summary_out = FRAGMENTATION_DIR / f"{sample}_library_summary.tsv"
    if all(p.exists() for p in (out_counts, out_fa, out_qc, summary_out)) and not force:
        print("[skip] library", sample)
        return

    rows = read_dict_rows(inp)
    seqs = dict(iter_fasta(TRUTH_DIR / f"{sample}_templates.fasta"))
    rng = np.random.default_rng(sample_seed(sample, 2000))
    pcr1 = np.array([int(row["pcr_copies"]) for row in rows], dtype=np.int64)
    lengths = np.array([int(row["length"]) for row in rows], dtype=np.int64)

    # порция ПЦР1: точная случайная доля пула без возвращения
    aliquot = sample_fraction_without_replacement(pcr1, PCR1_TO_FRAGMENTATION_FRACTION, rng)
    molecule_template = np.repeat(np.arange(len(rows)), aliquot)
    piece_template, piece_start, piece_end = fragment_molecules(
        molecule_template, lengths[molecule_template], rng
    )

    # одинаковые куски разных молекул — один вид куска с числом копий k
    span = int(lengths.max()) + 1
    species, input_copies = np.unique(
        (piece_template * span + piece_start) * span + piece_end, return_counts=True
    )
    species_template = species // (span * span)
    species_start = (species // span) % span
    species_end = species % span

    pcr2 = fixed_fold_amplification(input_copies, PCR2_FOLD, rng)
    selected = sample_fraction_without_replacement(pcr2, PCR2_TO_SEQUENCING_FRACTION, rng)
    pairs = selected * READ_PAIRS_PER_SELECTED_COPY

    fields = [
        "fragment_id", "template_id", "locus", "template_length",
        "fragment_start", "fragment_length", "fragment_input_copies",
        "library_pcr2_copies", "sequencing_selected_copies", "simulated_read_pairs",
    ]
    tmp = {path: Path(str(path) + ".tmp") for path in (out_counts, out_fa, out_qc)}
    with open(tmp[out_counts], "w", newline="") as counts_handle, \
         open(tmp[out_fa], "w") as fasta_handle, \
         open(tmp[out_qc], "w", newline="") as qc_handle:
        counts_writer = csv.writer(counts_handle, delimiter="\t")
        qc_writer = csv.DictWriter(qc_handle, fieldnames=fields, delimiter="\t")
        qc_writer.writeheader()
        for j in np.flatnonzero(pairs):
            row = rows[species_template[j]]
            start, end = int(species_start[j]), int(species_end[j])
            fragment_id = f"{row['template_id']}_{FRAGMENT_ID_TAG}_{start}_{end}"
            # в readcount_file InSilicoSeq передаётся число ридов, а не пар
            counts_writer.writerow([fragment_id, int(pairs[j]) * 2])
            fasta_handle.write(f">{fragment_id}\n{seqs[row['template_id']][start:end]}\n")
            qc_writer.writerow({
                "fragment_id": fragment_id,
                "template_id": row["template_id"],
                "locus": row["locus"],
                "template_length": int(lengths[species_template[j]]),
                "fragment_start": start,
                "fragment_length": end - start,
                "fragment_input_copies": int(input_copies[j]),
                "library_pcr2_copies": int(pcr2[j]),
                "sequencing_selected_copies": int(selected[j]),
                "simulated_read_pairs": int(pairs[j]),
            })
    for final, temporary in tmp.items():
        temporary.replace(final)

    n_input = int(aliquot.sum())
    with_reads = pairs > 0
    summary = {
        "sample": sample,
        "fragmentation_model": FRAGMENTATION_MODEL,
        "pcr1_templates": len(rows),
        "starting_copies_total": sum(int(row["starting_copies"]) for row in rows),
        "pcr1_fold": PCR1_FOLD,
        "pcr1_molecules_total": int(pcr1.sum()),
        "library_input_molecules": n_input,
        "library_aliquot_fraction_of_pcr1": round(n_input / int(pcr1.sum()), 6),
        "library_input_templates": int(np.count_nonzero(aliquot)),
        "fragmentation_pieces_total": 2 * n_input,
        "fragments_after_size_selection": int(len(piece_start)),
        "retained_fragments_per_input_molecule": round(len(piece_start) / n_input, 4) if n_input else 0,
        "fragment_species": int(len(species)),
        "fragment_species_with_duplicates": int((input_copies > 1).sum()),
        "pcr2_fold": PCR2_FOLD,
        "pcr2_pool_copies": int(pcr2.sum()),
        "pcr2_selected_copies": int(selected.sum()),
        "pcr2_selected_fraction": int(selected.sum()) / int(pcr2.sum()) if int(pcr2.sum()) else 0,
        "read_pairs": int(pairs.sum()),
        "templates_with_reads": int(len(np.unique(species_template[with_reads]))),
        "fragment_species_with_reads": int(with_reads.sum()),
        "size_selection_range_low_nt": SIZE_SELECTION_RANGE[0],
        "size_selection_range_high_nt": SIZE_SELECTION_RANGE[1],
        "size_selection_half_distance_nt": SIZE_SELECTION_HALF_DISTANCE_NT,
        "min_sequencable_fragment_nt": MIN_FRAGMENT_LENGTH_FOR_SEQUENCING_NT,
    }
    write_dict_rows(summary_out, [summary], list(summary))
    print(f"[{sample}] library: {summary}")


for sample in SAMPLES:
    run_pcr1_stage(sample)
    run_library_stage(sample)


## 12. Сводка библиотеки


In [ ]:
library_summary = [read_dict_rows(FRAGMENTATION_DIR / f"{sample}_library_summary.tsv")[0] for sample in SAMPLES]
out = QC_DIR / "library_summary.tsv"
write_dict_rows(out, library_summary, list(library_summary[0]))
print(*library_summary, sep="\n")
EXPECTED_PAIRS = {row["sample"]: int(row["read_pairs"]) for row in library_summary}


In [ ]:
# Класс восстановимости эталонного интервала V…J каждого шаблона по тому, что реально попало в FASTQ.
# В режиме ампликона InSilicoSeq: R1 = [start, start+R), R2 = [end-R, end); вся вставка остаётся физически связанной.
# Связь (bond) b между позициями b-1 и b:
#   read_overlap — связь попадает в один рид, и с обеих сторон остаётся не менее K нт.
#                  (собирается обычным перекрытием);
#   mate_only — связь лежит во фрагменте, но между ридами, в несеквенированном промежутке.
#               соединить её может только удлинение по паре ридов в TRUST4.
#   unlinked — связь не пересекается ни одним фрагментом: части шаблона физически
#              не связаны и не могут быть соединены без внешней информации.
RECONSTRUCT_MIN_FLANK_NT = 31  # грубый аналог минимального перекрытия (k-мера) сборщика, калибруемый

def _bond_mask(L, intervals):
    d = np.zeros(L + 2, dtype=np.int32)
    for a, b in intervals:
        a, b = max(a, 1), min(b, L - 1)
        if a <= b:
            d[a] += 1
            d[b + 1] -= 1
    return np.cumsum(d)[: L + 1] > 0

def classify_template(L, frags, R=TARGET_READ_LENGTH_NT, K=RECONSTRUCT_MIN_FLANK_NT, region=None):
    """frags: [(start, end)] фрагментов с ридами в координатах шаблона.

    region=(vj_start, vj_end): оценивается только эталонный интервал V…J
    (праймеры и leader/C-фланги не требуют сборки). Возвращает (class, read_covered_fraction).
    """
    reads = [iv for s, e in frags for iv in ((s, min(s + R, e)), (max(e - R, s), e))]
    if region is not None:
        a0, b0 = region
        clip = lambda ivs: [(max(a, a0) - a0, min(b, b0) - a0) for a, b in ivs if min(b, b0) > max(a, a0)]
        L, frags, reads = b0 - a0, clip(frags), clip(reads)
    if not frags:
        return "no_reads", 0.0
    cov = np.zeros(L + 1, dtype=np.int32)
    for a, b in reads:
        cov[a] += 1
        cov[b] -= 1
    covered = float((np.cumsum(cov)[:L] > 0).mean())
    if covered < 1.0:
        return "partial_read_coverage", covered
    by_frag = _bond_mask(L, [(s + K, e - K) for s, e in frags])
    by_read = _bond_mask(L, [(a + K, b - K) for a, b in reads])
    bonds = slice(K, L - K + 1)  # связи ближе K к краям шаблона не требуют соединения
    if not by_frag[bonds].all():
        return "full_coverage_unlinked", covered
    if not by_read[bonds].all():
        return "full_coverage_mate_linked_only", covered
    return "full_coverage_read_overlap", covered

# Самопроверка: фрагмент 400 нт, парные риды PE150.
assert classify_template(400, [])[0] == "no_reads"
assert classify_template(400, [(0, 200), (200, 400)])[0] == "full_coverage_unlinked"
assert classify_template(300, [(0, 200), (100, 300)])[0] == "full_coverage_read_overlap"
assert classify_template(400, [(0, 400)])[0] == "partial_read_coverage"          # середина между R1 и R2 не прочитана
assert classify_template(300, [(0, 300)])[0] == "full_coverage_mate_linked_only"  # R1 и R2 стыкуются без перекрытия
assert classify_template(420, [(0, 420), (100, 320)])[0] == "full_coverage_mate_linked_only"
# фланги вне V…J не учитываются; фрагмент только в праймере = нет ридов в V…J
assert classify_template(500, [(0, 300), (200, 500)], region=(50, 450))[0] == "full_coverage_mate_linked_only"
assert classify_template(500, [(0, 40)], region=(50, 450))[0] == "no_reads"

def write_reconstructability_qc(sample):
    frags = defaultdict(list)
    for r in read_dict_rows(ALLOCATION_DIR / f"{sample}_allocation.tsv"):
        if int(r["simulated_read_pairs"]) > 0:
            s = int(r["fragment_start"])
            frags[r["template_id"]].append((s, s + int(r["fragment_length"])))
    rows, summary = [], defaultdict(lambda: defaultdict(int))
    for t in load_dedup_templates(sample):
        cls, cov = classify_template(t["length"], frags.get(t["template_id"], []),
                                     region=(t["vj_start"], t["vj_end"]))
        rows.append({"template_id": t["template_id"], "locus": t["locus"], "length": t["length"],
                     "unique_umis": t["unique_umis"],
                     "fragments_with_reads": len(frags.get(t["template_id"], [])),
                     "read_covered_fraction": round(cov, 4), "reconstructability": cls, "verified": t["verified"]})
        if t["verified"]:  # знаменатели оценки сборщиков — только проверенные шаблоны
            summary[t["locus"]][cls] += 1
    write_dict_rows(QC_DIR / f"{sample}_template_reconstructability.tsv", rows, list(rows[0]))
    classes = ["no_reads", "partial_read_coverage", "full_coverage_unlinked",
               "full_coverage_mate_linked_only", "full_coverage_read_overlap"]
    out = []
    for locus in sorted(summary) + ["ALL"]:
        c = {k: (sum(summary[l][k] for l in summary) if locus == "ALL" else summary[locus][k]) for k in classes}
        n = sum(c.values())
        out.append({"sample": sample, "locus": locus, "templates": n, **c,
                    **{f"{k}_frac": round(c[k] / n, 4) for k in classes}})
    write_dict_rows(QC_DIR / f"{sample}_reconstructability_summary.tsv", out, list(out[0]))
    print(*out, sep="\n")

for sample in SAMPLES:
    write_reconstructability_qc(sample)


## 13. Генерация ридов точной длины PE150


In [ ]:
def crop_fastq(in_path,out_path,length):
    tmp=Path(str(out_path)+".tmp")
    with open_text(in_path,"rt") as src,gzip.open(tmp,"wt") as dst:
        n=0
        while True:
            h=src.readline()
            if not h: break
            s=src.readline().rstrip("\n\r"); p=src.readline(); q=src.readline().rstrip("\n\r")
            if len(s)<length or len(q)<length: raise ValueError(f"short read in {in_path}: {len(s)}")
            dst.write(h); dst.write(s[:length]+"\n"); dst.write(p); dst.write(q[:length]+"\n"); n+=1
    tmp.replace(out_path); return n

def run_iss_generate(sample,force=FORCE):
    fa=ALLOCATION_DIR/f"{sample}_selected_fragments.fasta"; counts=ALLOCATION_DIR/f"{sample}_read_counts.tsv"; native_prefix=FASTQ_NATIVE_DIR/sample; final_prefix=FASTQ_DIR/sample; ext=".fastq.gz"
    nr1=Path(str(native_prefix)+f"_R1{ext}"); nr2=Path(str(native_prefix)+f"_R2{ext}"); r1=Path(str(final_prefix)+f"_R1{ext}"); r2=Path(str(final_prefix)+f"_R2{ext}")
    if r1.exists() and r2.exists() and not force: print("[skip]",sample); return
    cmd=["iss","generate","--genomes",str(fa),"--readcount_file",str(counts),"--sequence_type",SEQUENCE_TYPE,"--model",ISS_PROFILE_NAME,"--cpus",str(ISS_GENERATOR_CPUS),"--output",str(native_prefix),"--seed",str(sample_seed(sample,5000)),"--compress"]
    run_with_heartbeat(cmd,LOGS_DIR/f"{sample}_iss_novaseq.log")
    n1=crop_fastq(nr1,r1,TARGET_READ_LENGTH_NT); n2=crop_fastq(nr2,r2,TARGET_READ_LENGTH_NT); assert n1==n2==EXPECTED_PAIRS[sample]
    print("NovaSeq native PE151 -> exact PE150:",n1,"pairs")
for sample in SAMPLES: run_iss_generate(sample)


## 14. Итоговая проверка


In [ ]:
final_rows=[]
for sample in SAMPLES:
    r1=FASTQ_DIR/f"{sample}_R1.fastq.gz"; r2=FASTQ_DIR/f"{sample}_R2.fastq.gz"; n1=count_fastq(r1); n2=count_fastq(r2)
    lengths1=sorted({len(s) for s in iter_fastq(r1)}); lengths2=sorted({len(s) for s in iter_fastq(r2)})
    final_rows.append({"branch":BRANCH_NAME,"sample":sample,"expected_pairs":EXPECTED_PAIRS[sample],"R1_reads":n1,"R2_reads":n2,"R1_lengths":",".join(map(str,lengths1)),"R2_lengths":",".join(map(str,lengths2)),"valid":n1==n2==EXPECTED_PAIRS[sample] and lengths1==lengths2==[TARGET_READ_LENGTH_NT]})
out=QC_DIR/"final_qc.tsv"; write_dict_rows(out,final_rows,list(final_rows[0])); assert all(r["valid"] for r in final_rows); print(*final_rows,sep="\n")


## Интерпретация и научные ограничения

- Биологический эталон формируется из AIRR-записей после фильтрации аннотаций.
- Сохраняется наблюдаемая последовательность V…J; germline-последовательность её не заменяет, поэтому реальные SHM остаются в данных.
- Шаблон — физический ампликон PCR1 (V-leader праймер + склеенный рид + C-праймер); эталон для оценки сборки — интервал `vj_start…vj_end`. UMI и Illumina-хвосты в шаблон не включены; вырожденные позиции праймеров разрешены случайно, но воспроизводимо.
- Каждый UMI — одна молекула; её V…J — консенсус ридов UMI. Шаблоны без проверенного консенсуса (≈80% UMI прочитаны одним ридом) симулируются, но в оценку сборщиков не входят.
- Численность UMI задаёт исходные копии. Записи с отсутствующим или некорректным UMI и неоднозначными N исключаются и учитываются отдельно.
- ПЦР1 и ПЦР2 заданы фиксированной кратностью общего числа молекул (×20 и ×3,5) без циклов и эффективности; новые копии раздаются случайно пропорционально числу копий. Замены полимеразы и химеры не моделируются.
- Бюджета ридов нет: число пар равно числу копий, отобранных из пула ПЦР2 на секвенирование; доли IGH/IGK/IGL определяются числом отобранных копий каждого локуса.
- Встроенная модель InSilicoSeq NovaSeq является общим профилем платформы: ISS создаёт PE151, после чего оба рида детерминированно обрезаются до PE150.
- Смешанный IGH+IGK+IGL набор является вычислительным тестом без нативного спаривания тяжёлой и лёгкой цепей.
